# PMCC Backtest

Poor Man's Covered Call (long deep-ITM LEAPS + short near-term OTM call) on SPY.

**Prerequisites**: Pre-downloaded data via optopsy-data:
```bash
EODHD_API_KEY=... optopsy-data download SPY        # options
optopsy-data download SPY -s                       # stock OHLCV
```

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import altair as alt

alt.renderers.enable("jupyter")

from options_strategies.pmcc import PmccConfig, run_pmcc
from options_strategies.shared import load_pmcc_data
from backtest_charts import plot_equity_curve, plot_cumulative_pnl, plot_pnl_distribution, plot_exit_breakdown, plot_dashboard, plot_portfolio

## Configuration

In [2]:
config = PmccConfig(
    symbol="SPY",
    capital=100_000.0,
    quantity=1,
    multiplier=100,
    max_positions=1,
    # LEAPS: deep-ITM, far expiry
    leaps_delta=0.80,
    leaps_delta_min=0.75,
    leaps_delta_max=0.80,
    leaps_max_entry_dte=365,
    leaps_exit_dte=30,
    # Short call: near-term, 80% profit exit
    short_delta=0.30,
    short_delta_min=0.20,
    short_delta_max=0.30,
    short_max_entry_dte=30,
    short_exit_dte=7,
    short_take_profit=0.8,
    short_stop_loss=-0.11,
    leaps_weight=0.6,
    short_weight=0.4,
)
config

PmccConfig(symbol='SPY', capital=100000.0, quantity=1, multiplier=100, max_positions=1, start_date=None, end_date=None, expiration_type='monthly', leaps_delta=0.8, leaps_delta_min=0.75, leaps_delta_max=0.8, leaps_max_entry_dte=365, leaps_exit_dte=30, leaps_max_hold_days=None, short_delta=0.3, short_delta_min=0.2, short_delta_max=0.3, short_max_entry_dte=30, short_exit_dte=7, short_take_profit=0.8, short_stop_loss=-0.11, short_max_hold_days=None, leaps_weight=0.6, short_weight=0.4)

## Load Data

In [3]:
print(f"Loading data for {config.symbol}\u2026")
options, stock = load_pmcc_data(
    config.symbol,
    start_date=config.start_date,
    end_date=config.end_date,
    expiration_type=config.expiration_type,
)
print(f"  Options: {len(options):,} rows")
print(f"  Stock:   {len(stock):,} rows")

Loading data for SPY…
  Options: 2,922,370 rows
  Stock:   8,428 rows


## Run Backtest

In [4]:
print("Running PMCC backtest\u2026")
result = run_pmcc(options, stock, config)

Running PMCC backtest…


e:\TMP\trade-system\.venv\Lib\site-packages\empyrical\stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))


## Summary

In [5]:
s = result.summary
print("\u2550\u2550\u2550 PMCC Portfolio Summary \u2550\u2550\u2550")
print(f"  Total trades:    {s.get('total_trades', 0)}")
print(f"  Win rate:        {s.get('win_rate', 0):.1%}")
print(f"  Total P&L:       ${s.get('total_pnl', 0):,.2f}")
print(f"  Max drawdown:    {s.get('max_drawdown', 0):.2%}")
print(f"  Sharpe ratio:    {s.get('sharpe_ratio', 0):.2f}")
print(f"  Sortino ratio:   {s.get('sortino_ratio', 0):.2f}")
print(f"  Profit factor:   {s.get('profit_factor', 0):.2f}")
print(f"  Avg days held:   {s.get('avg_days_in_trade', 0):.1f}")

═══ PMCC Portfolio Summary ═══
  Total trades:    100
  Win rate:        37.0%
  Total P&L:       $6,584.50
  Max drawdown:    -6.25%
  Sharpe ratio:    0.41
  Sortino ratio:   0.80
  Profit factor:   1.40
  Avg days held:   9.4


## Per-Leg Results

In [6]:
for name, leg in result.leg_results.items():
    ls = leg.summary
    print(f"\n  \u2500\u2500 {name} leg \u2500\u2500")
    print(
        f"    Trades: {ls.get('total_trades', 0)}  "
        f"Win rate: {ls.get('win_rate', 0):.1%}  "
        f"P&L: ${ls.get('total_pnl', 0):,.2f}"
    )
    tl = leg.trade_log
    if not tl.empty and "exit_type" in tl.columns:
        for exit_type in sorted(tl["exit_type"].unique()):
            count = (tl["exit_type"] == exit_type).sum()
            print(f"    {exit_type}: {count}")


  ── leaps leg ──
    Trades: 8  Win rate: 50.0%  P&L: $8,324.50
    expiration: 8

  ── short_call leg ──
    Trades: 92  Win rate: 35.9%  P&L: $-1,740.00
    expiration: 2
    stop_loss: 59
    take_profit: 31


## Trade Log Sample

In [7]:
if not result.trade_log.empty:
    result.trade_log.head(10)

## Visualizations

### Equity Curve

In [8]:
plot_equity_curve(result, config.capital)

alt.LayerChart(...)

### Cumulative P&L by Leg

In [9]:
plot_cumulative_pnl(result)

alt.Chart(...)

### Per-Trade P&L Distribution

In [10]:
plot_pnl_distribution(result)

alt.Chart(...)

### Exit Type Breakdown

In [11]:
plot_exit_breakdown(result)

alt.Chart(...)

### Full Dashboard

In [12]:
plot_dashboard(result, config.capital)

alt.VConcatChart(...)